In [ ]:
import os
import json
import uuid
import time
import asyncio
import random
import string
import pandas as pd

from typing import List, Optional, Literal, Dict, Any
from pydantic import BaseModel, Field, conint
from langchain_openai import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from tqdm.auto import tqdm
from pathlib import Path
from glob import glob

In [ ]:
JSONL_INPUT_DIR = r"D:\Oxford\Master Thesis\Data for DNA\Corpus\output\dedup"
OUTPUT_DIR = r"D:\Oxford\Master Thesis\Data for DNA\Corpus\output\statementExtraction"

## TODO
1. Add consistent input / output formats to run everything easier

In [ ]:
SYSTEM_PROMPT = """
# System Prompt: Statement Extraction, Attribution, and De-contextualisation

## Task

You are given a news article or other text.

Your task is to:

1. Segment the text into statements
2. Attribute each statement to exactly one actor
3. Preserve all actor information available in the text and metadata
4. Produce a de-contextualised version of each statement
5. Explicitly record which mentions in the statement required de-contextualisation and how they were resolved

Return the result as a single JSON object with a top-level `statements` array.

---

## 1. Definition of a Statement

A statement is a **minimal text segment expressing one coherent main idea** about actors, actions, events, states of affairs, or developments.

An article consists entirely of statements.

Your task is to determine **where one statement ends and the next begins**.

A statement may include subordinate material such as reasons, elaborations, contrasts, conditions, examples, comparisons, and implicit premises **as long as they support, qualify, or specify the same main idea rather than introducing a separate independent main idea**.

---

## 2. Segmentation Rules

### 2.1 Core Principle

Segment based on **main-idea coherence** and **source coherence**:

> A statement must express **one main idea** and be attributable to **one actor**.
> 

---

### 2.2 Treat a span as ONE statement if

- It expresses **one main idea**
- Additional clauses only elaborate, support, qualify, exemplify, contrast with, or specify that same main idea
- Multiple predicates describe the same object within the same framing function
- A paraphrase and a quote express the same idea without adding new meaning

---

### 2.3 Split into separate statements when ANY of the following apply

1. **Change of main idea**
    - A new independent proposition, development, or argumentative move is introduced
2. **Change of actor (MANDATORY)**
    - Different actors are responsible for different parts of the span
3. **Evidence vs. stance**
    - Reporting findings, data, or surveys must be separated from positions, evaluations, or judgments, even when they come from the same actor
4. **New semantic content**
    - A clause introduces a distinct cause, evaluation, justification, consequence, target, or other independent main idea
5. **Distinct direct quotes**
    - Each direct quote expressing a distinct main idea is a separate statement

---

### 2.4 Do NOT split when

1. **Same idea (elaboration)**
    - Additional detail, specification, explanation, or support belongs to the same main idea
2. **Same framing function**
    - Multiple predicates jointly frame the same object or development
3. **Paraphrase + quote equivalence**
    - Merge a quote with surrounding text only if all of the following hold:
        - same actor
        - same main idea
        - the quote is a restatement or mild intensification
        - no new target, cause, evaluation, justification, or consequence is introduced

**Test:**

If removing the quote does not remove any new meaning, merging is allowed.

---

### 2.5 Default Rule

When in doubt:

- Prefer **splitting** if:
    - there may be more than one actor
    - there may be more than one independent main idea

---

## 3. Attribution Rules

### 3.1 Core Principle

Each statement must be attributed to **exactly one actor**.

This actor should be represented with the **most specific de-contextualised information available in the text or metadata**, while preserving any available affiliation information in the structured actor fields.

Do **not** prematurely collapse actors to organization level if the text provides richer actor information.

---

### 3.2 Attribution Hierarchy

Use the following order:

1. **Explicit named actor**
    - person, organization, or other clearly identifiable source named in or near the span
2. **Actor recoverable from immediate context**
    - pronoun, title, office, or shorthand reference resolvable with minimal inference
3. **Diffuse actor**
    - collective but non-specific source explicitly present in the text
    - examples: “experts”, “critics”, “start-ups”, “officials”
4. **Article author**
    - if the statement cannot be attributed to any other actor, assign it to the article author using article metadata
    - if an individual author is provided in metadata, use that individual as actor and store the outlet in `affiliated_orgs`
    - if no individual author is provided, use the domain name as actor without language path specification, e.g. "gematik" for the domain gematik.de, but keep detailed path specification like bundestag.de/ausschuesse/gesundheit

---

### 3.3 Article Author Default Rule (CRITICAL)

> Every statement that is not attributable to any actor other than the article author must be attributed to the article author or domain name.
> 

Unattributed narration, framing, or description is **not left unattributed**.

It must be assigned to the article author from the article metadata.

**Article-author attribution rule:**

- If metadata contains an individual author, use that person as the actor.
- Store the publishing outlet in `affiliated_orgs`.
- Set `person_type` to `ARTICLE_AUTHOR`.
- If no individual author is available, use the domain name as actor without language path specification, e.g. "gematik" for the domain gematik.de, but keep detailed path specification like bundestag.de/ausschuesse/gesundheit

---

### 3.4 Reporting Rule

Transform reported speech as follows:

> “X said that Y”
> 
> 
> → Statement = Y
> 
> → Actor = X
> 

If multiple independent ideas are reported, split them into separate statements.

---

### 3.5 Preserve Maximal Actor Information

Use the actor schema to preserve all available actor information:

- `name`: most specific de-contextualised actor name available
- `entity_kind`: `PERSON`, `ORGANIZATION`, `OTHER`, or `UNKNOWN`
- `org_type`: only if the actor is an organization or can clearly be typed organizationally
- `person_type`: only if the actor is a person
- `role_or_title`: preserve a role/title only if it is stated in the text or metadata; otherwise use `null`
- `affiliated_orgs`: preserve any explicitly stated or clearly inferable affiliations from the text or metadata

Examples:

If the text says:

“Alice Oh, a computer science professor at KAIST, said the law was imperfect.”

```
{{
  "name":"Alice Oh",
  "entity_kind":"PERSON",
  "org_type":null,
  "person_type":"ACADEMIC",
  "role_or_title":"computer science professor",
  "affiliated_orgs": [
    {{
      "affiliated_org_name":"Korea Advanced Institute of Science and Technology",
      "affiliated_org_type":"ACADEMIC_INSTITUTION",
      "affiliated_org_person_type":"ACADEMIC"
    }}
  ]
}}
```

If the text says:

“Government officials said the law would support innovation.”

```
{{
  "name":"South Korean government officials",
  "entity_kind":"ORGANIZATION",
  "org_type":"GOVERNMENT",
  "person_type":null,
  "role_or_title":"government officials",
  "affiliated_orgs": []
}}
```

If metadata says:

- author: `Raphael Rashidi`
- source: `The Guardian`

then unattributed narration should use:

```
{{
  "name":"Raphael Rashidi",
  "entity_kind":"PERSON",
  "org_type":null,
  "person_type":"ARTICLE_AUTHOR",
  "role_or_title":null,
  "affiliated_orgs": [
    {{
      "affiliated_org_name":"The Guardian",
      "affiliated_org_type":"MEDIA",
      "affiliated_org_person_type":"JOURNALIST"
    }}
  ]
}}
```

If the text says:

“Mark Zuckerberg said the company would release the model next year.”

```
{{
  "name":"Mark Zuckerberg",
  "entity_kind":"PERSON",
  "org_type":null,
  "person_type":"EXECUTIVE",
  "role_or_title":null,
  "affiliated_orgs": [
    {{
      "affiliated_org_name":"Meta",
      "affiliated_org_type":"BIG_TECH",
      "affiliated_org_person_type":"EXECUTIVE"
    }}
  ]
}}
```

---

### 3.6 Strict Constraint

> A statement must not contain content attributable to multiple actors.
> 

If it does, split it.

---

### 3.7 Joint Statement Edge Case

If a statement is clearly attributed to **multiple actors jointly making the same statement**, extract the statement **n times**, where **n** is the number of actors.

- Each extracted item should contain the same statement span and same statement content
- Each item should assign the statement to **one** of the actors

Use this only when the text clearly presents a genuinely joint statement or jointly held position.

---

## 4. De-contextualisation Rules

### 4.1 Goal

Each statement must be fully interpretable on its own, without relying on the original text.

De-contextualisation must be made **explicit** in the output.

You must:

1. keep the original statement span verbatim in `span_text`
2. identify all mentions in the statement that require de-contextualisation
3. provide the de-contextualised resolution of each such mention
4. use those resolutions to produce a de-contextualised `central_claim`

---

### 4.2 Verbatim Span Rule (CRITICAL)

> `span_text` must match the source text exactly.
> 

Copy the statement span exactly as it appears in the source text, preserving:

- wording
- spelling
- punctuation
- quotation marks
- capitalization

Do **not** normalize, correct, paraphrase, trim, or silently rewrite `span_text` in any way.

Only `mentions_to_decontextualize` and `central_claim` may contain resolved or expanded formulations.

---

### 4.3 De-contextualise the Statement

For each statement, identify ambiguous or context-dependent mentions such as:

- pronouns: “it”, “this”, “they”, “she”, “he”
- shorthand references: “the law”, “the company”, “the regulator”
- vague noun phrases: “the decision”, “the move”, “the proposal”
- elliptical or implicit objects: “do this”, “be the first”, “that approach”

For each such mention:

- extract the original mention
- provide the most explicit resolution supported by the full text
- use only information available in the text or metadata where explicitly allowed for article-author attribution
- do not introduce external knowledge
- do not over-interpret

---

### 4.4 De-contextualise the Actor

You must also de-contextualise the actor.

This includes:

- resolving names, pronouns, titles, and shorthand references
- preserving person-level specificity where available
- preserving organization-level affiliations where available
- assigning the article author if no other actor is attributable

If the actor is assigned via metadata rather than textual mention:

- use metadata to fill the actor fields
- set `mention_in_article` to `null`

---

### 4.5 Constraints

- Do not add information not supported by the text or metadata
- Do not guess when resolution is uncertain
- Keep transformation minimal, but sufficient for standalone clarity

---

### 4.6 Missingness and `UNKNOWN` Rule

Use `null` and `UNKNOWN` differently.

- Use `null` when a field is **not applicable** or **not available from the text/metadata**
- Use `UNKNOWN` **only for ontology-coded fields** when the field is applicable in principle but the correct ontology value cannot be determined confidently

Apply this rule as follows:

- **Free-text fields** must use `null`, never `UNKNOWN`
    - `name`
    - `role_or_title`
    - `mention_in_article`
    - `affiliated_org_name`
- **Ontology-coded fields** may use `UNKNOWN` when applicable but unresolved
    - `entity_kind`
    - `org_type`
    - `person_type`
    - `affiliated_org_type`
    - `affiliated_org_person_type`

Examples:

- If the actor is a person, `org_type` = `null`
- If the actor is an organization, `person_type` = `null`
- If an actor exists but it is unclear whether it is a person or organization, `entity_kind` = `UNKNOWN`
- If a person is clearly present but their person-type category cannot be determined, `person_type` = `UNKNOWN`
- If no canonical actor name can be resolved, `name` = `null`
- If metadata-based attribution is used and there is no textual mention, `mention_in_article` = `null`

---

### 4.7 Principle

> Minimal transformation for maximum clarity
> 

---

## 5. Attribution Confidence

### Attribution Confidence (`attribution_confidence`)

| Level | Label | Description |
| --- | --- | --- |
| **4** | Certain | The actor is explicitly named in or immediately adjacent to the span, or is determinable with certainty from metadata in the case of article-author attribution. |
| **3** | Probable | The actor is identifiable from immediate context with minimal inference. |
| **2** | Possible | The actor requires moderate inference from broader text context; some ambiguity remains. |
| **1** | Speculative | The actor attribution is largely inferred; meaningful ambiguity remains. |

---

## 6. Output Format

Return a single JSON object with the following schema:

```
{{
  "statements": [
    {{
      "span_text":"<exact text of the statement span as it appears in the text>",
      "span_char_start":"<integer index of first character, or null if unknown>",
      "span_char_end":"<integer index of last character (exclusive), or null if unknown>",
      "actor": {{
        "name":"<de-contextualised name>",
        "entity_kind":"<PERSON | ORGANIZATION | OTHER | UNKNOWN>",
        "org_type":"<one org ontology code, or null>",
        "person_type":"<one person ontology code, or null>",
        "role_or_title":"<short free-text role/title, or null>",
        "affiliated_orgs": [
          {{
            "affiliated_org_name":"<canonical affiliated organization name, or null>",
            "affiliated_org_type":"<org ontology code, or null>",
            "affiliated_org_person_type":"<person type ontology code indicating the actor's role within that organization, or null>"
          }}
        ]
      }},
      "attribution": {{
        "mention_in_article":"<how the actor is referred to in or near the span, or null if actor is assigned only via metadata>",
        "attribution_type":"<DIRECT_QUOTE | INDIRECT_SPEECH | PARAPHRASED_POSITION | PRONOUN_REFERENCE | OTHER>",
        "attribution_confidence":"<1 | 2 | 3 | 4>"
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"<original ambiguous or context-dependent mention>",
            "resolved_text":"<de-contextualised resolution>"
          }}
        ],
        "central_claim":"<de-contextualised version of the statement>"
      }},
      "reason":"<brief explanation of why this span is one statement, how it was attributed, and any important de-contextualisation decisions>"
    }}
  ]
}}
```

---

## 7. Ontologies

### Org Type Ontology

`STARTUP` · `BIG_TECH` · `INCUMBENT` · `NGO_CSO` · `VENTURE_CAPITAL` · `OTHER_FINANCIAL_INDUSTRY` · `INDUSTRY_ASSOCIATION` · `STARTUP_ASSOCIATION` · `GOVERNMENT` · `NATIONAL_SECURITY` · `INTERNATIONAL_SECURITY` · `GEOPOLITICAL_ENTITY` · `ACADEMIC_INSTITUTION` · `MEDIA` · `POLITICAL_PARTY` · `REGULATOR` · `SME_BUT_NOT_STARTUP` · `SOCIAL_MOVEMENT` · `CIVIL_SOCIETY_INTEREST_GROUP` · `PROFESSIONAL_SERVICES_FIRM` · `OTHER_ORG_TYPE` · `UNKNOWN`

### Person Type Ontology

`EXECUTIVE` · `JOURNALIST` · `OTHER_EXPERT` · `ACADEMIC` · `INDEPENDENT_RESEARCHER` · `POLITICIAN` · `BUSINESS_PEOPLE`· `SUPERVISORY_BOARD` · `ACTIVIST` · `PUBLIC_SERVANT` · `PROFESSIONAL` · `UNKNOWN` · `ARTICLE_AUTHOR`

---

## 8. Entity Resolution Rules

- Use the entire text to resolve pronouns and shorthand references
- Use article metadata for article-author attribution
- All fields in the output must be de-contextualised
- If resolution is uncertain, use `null` for free-text fields and `UNKNOWN` only for applicable ontology-coded fields
- Do not guess

---

## 9. Example (South Korea AI Law)

### Input (excerpt)

“The legislation is being billed as the ‘world’s first’ to be fully enforced by a country, and central to South Korea’s ambition to become one of the world’s three leading AI powers alongside the US and China. Government officials maintain the law is 80–90% focused on promoting industry rather than restricting it. Alice Oh, a computer science professor at KAIST, said that while the law was not perfect, it was intended to evolve without stifling innovation. However a survey in December from the Startup Alliance found that 98% of AI startups were unprepared for compliance. Its co-head, Lim Jung-wook, said frustration was widespread. ‘There’s a bit of resentment,’ he said. ‘Why do we have to be the first to do this?’”

Article metadata:

- source: `The Guardian`
- org_type: `MEDIA`
- source_type: `NEWS_ARTICLE`
- author: `Raphael Rashidi`
- date: `29.01.2026`

### Output

```
{{
  "statements": [
    {{
      "span_text":"The legislation is being billed as the ‘world’s first’ to be fully enforced by a country, and central to South Korea’s ambition to become one of the world’s three leading AI powers alongside the US and China.",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"Raphael Rashidi",
        "entity_kind":"PERSON",
        "org_type":null,
        "person_type":"ARTICLE_AUTHOR",
        "role_or_title":null,
        "affiliated_orgs": [
          {{
            "affiliated_org_name":"The Guardian",
            "affiliated_org_type":"MEDIA",
            "affiliated_org_person_type":"JOURNALIST"
          }}
        ]
      }},
      "attribution": {{
        "mention_in_article":null,
        "attribution_type":"OTHER",
        "attribution_confidence":4
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"The legislation",
            "resolved_text":"the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }},
          {{
            "mention_text":"the US",
            "resolved_text":"the United States"
          }}
        ],
        "central_claim":"The AI Basic Act, South Korea’s 2026 comprehensive AI regulation, is being described as the world’s first AI law to be fully enforced by a country and as central to South Korea’s ambition to become one of the world’s three leading AI powers alongside the United States and China."
      }},
      "reason":"This span expresses one coherent framing idea and is not attributable to any actor other than the article author, so it is assigned to the individual article author from metadata."
    }},
    {{
      "span_text":"Government officials maintain the law is 80–90% focused on promoting industry rather than restricting it.",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"South Korean government officials",
        "entity_kind":"ORGANIZATION",
        "org_type":"GOVERNMENT",
        "person_type":null,
        "role_or_title":"government officials",
        "affiliated_orgs": []
      }},
      "attribution": {{
        "mention_in_article":"Government officials",
        "attribution_type":"INDIRECT_SPEECH",
        "attribution_confidence":3
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"the law",
            "resolved_text":"the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }}
        ],
        "central_claim":"South Korean government officials state that the AI Basic Act, South Korea’s 2026 comprehensive AI regulation, is 80–90% focused on promoting industry rather than restricting it."
      }},
      "reason":"This is one indirect-speech statement with one main idea. The actor is resolved from the article context as government officials associated with South Korea."
    }},
    {{
      "span_text":"while the law was not perfect, it was intended to evolve without stifling innovation.",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"Alice Oh",
        "entity_kind":"PERSON",
        "org_type":null,
        "person_type":"ACADEMIC",
        "role_or_title":"computer science professor",
        "affiliated_orgs": [
          {{
            "affiliated_org_name":"Korea Advanced Institute of Science and Technology",
            "affiliated_org_type":"ACADEMIC_INSTITUTION",
            "affiliated_org_person_type":"ACADEMIC"
          }}
        ]
      }},
      "attribution": {{
        "mention_in_article":"Alice Oh",
        "attribution_type":"INDIRECT_SPEECH",
        "attribution_confidence":4
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"the law",
            "resolved_text":"the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }},
          {{
            "mention_text":"it",
            "resolved_text":"the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }}
        ],
        "central_claim":"Alice Oh states that the AI Basic Act, South Korea’s 2026 comprehensive AI regulation, was not perfect but was intended to evolve without stifling innovation."
      }},
      "reason":"This span contains one main idea attributed explicitly to Alice Oh. Person-level actor information is preserved, and organizational affiliation is stored separately."
    }},
    {{
      "span_text":"a survey in December from the Startup Alliance found that 98% of AI startups were unprepared for compliance.",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"Startup Alliance",
        "entity_kind":"ORGANIZATION",
        "org_type":"INDUSTRY_ASSOCIATION",
        "person_type":null,
        "role_or_title":null,
        "affiliated_orgs": []
      }},
      "attribution": {{
        "mention_in_article":"the Startup Alliance",
        "attribution_type":"OTHER",
        "attribution_confidence":4
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"compliance",
            "resolved_text":"compliance with the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }}
        ],
        "central_claim":"A survey from the Startup Alliance found that 98% of AI startups were unprepared for compliance with the AI Basic Act, South Korea’s 2026 comprehensive AI regulation."
      }},
      "reason":"This is an evidence statement attributed to the organization that conducted or reported the survey. It is kept separate from later evaluative statements."
    }},
    {{
      "span_text":"frustration was widespread. ‘There’s a bit of resentment,’",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"Lim Jung-wook",
        "entity_kind":"PERSON",
        "org_type":null,
        "person_type":"EXECUTIVE",
        "role_or_title":"co-head",
        "affiliated_orgs": [
          {{
            "affiliated_org_name":"Startup Alliance",
            "affiliated_org_type":"INDUSTRY_ASSOCIATION",
            "affiliated_org_person_type":"EXECUTIVE"
          }}
        ]
      }},
      "attribution": {{
        "mention_in_article":"Its co-head, Lim Jung-wook",
        "attribution_type":"DIRECT_QUOTE",
        "attribution_confidence":4
      }},
      "content": {{
        "mentions_to_decontextualize": [],
        "central_claim":"Lim Jung-wook states that frustration was widespread and that there was a bit of resentment."
      }},
      "reason":"The paraphrase and quote are merged because they express the same main idea, are attributable to the same actor, and the quote adds no new independent meaning."
    }},
    {{
      "span_text":"‘Why do we have to be the first to do this?’",
      "span_char_start":null,
      "span_char_end":null,
      "actor": {{
        "name":"Lim Jung-wook",
        "entity_kind":"PERSON",
        "org_type":null,
        "person_type":"EXECUTIVE",
        "role_or_title":"co-head",
        "affiliated_orgs": [
          {{
            "affiliated_org_name":"Startup Alliance",
            "affiliated_org_type":"INDUSTRY_ASSOCIATION",
            "affiliated_org_person_type":"EXECUTIVE"
          }}
        ]
      }},
      "attribution": {{
        "mention_in_article":"he",
        "attribution_type":"DIRECT_QUOTE",
        "attribution_confidence":4
      }},
      "content": {{
        "mentions_to_decontextualize": [
          {{
            "mention_text":"we",
            "resolved_text":"AI startups represented by the Startup Alliance"
          }},
          {{
            "mention_text":"the first",
            "resolved_text":"the first AI startups required to comply with the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }},
          {{
            "mention_text":"do this",
            "resolved_text":"comply with the AI Basic Act, South Korea’s 2026 comprehensive AI regulation"
          }}
        ],
        "central_claim":"Lim Jung-wook questions why AI startups represented by the Startup Alliance have to be the first AI startups required to comply with the AI Basic Act, South Korea’s 2026 comprehensive AI regulation."
      }},
      "reason":"This direct quote expresses a distinct main idea and therefore forms a separate statement. Pronouns and elliptical references are explicitly resolved."
    }}
  ]
}}
```

## Final Reminder

- One statement = one main idea + one actor
- Preserve `span_text` exactly as it appears in the source text
- Make de-contextualisation explicit in structured form
- Preserve maximal actor information
- Assign unattributed narration to the article author
- Prefer splitting over merging when uncertain
"""



In [ ]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field, conint


# --- Ontology literals ---

EntityKind = Literal["PERSON", "ORGANIZATION", "OTHER", "UNKNOWN"]

OrgType = Literal[
    "STARTUP",
    "BIG_TECH",
    "INCUMBENT",
    "NGO_CSO",
    "VENTURE_CAPITAL",
    "OTHER_FINANCIAL_INDUSTRY",
    "INDUSTRY_ASSOCIATION",
    "STARTUP_ASSOCIATION",
    "GOVERNMENT",
    "NATIONAL_SECURITY",
    "INTERNATIONAL_SECURITY",
    "GEOPOLITICAL_ENTITY",
    "ACADEMIC_INSTITUTION",
    "MEDIA",
    "POLITICAL_PARTY",
    "REGULATOR",
    "SME_BUT_NOT_STARTUP",
    "SOCIAL_MOVEMENT",
    "CIVIL_SOCIETY_INTEREST_GROUP",
    "PROFESSIONAL_SERVICES_FIRM",
    "OTHER_ORG_TYPE",
    "UNKNOWN",
]

PersonType = Literal[
    "EXECUTIVE",
    "JOURNALIST",
    "OTHER_EXPERT",
    "ACADEMIC",
    "INDEPENDENT_RESEARCHER",
    "POLITICIAN",
    "BUSINESS_PEOPLE",
    "SUPERVISORY_BOARD",
    "ACTIVIST",
    "PUBLIC_SERVANT",
    "PROFESSIONAL",
    "UNKNOWN",
]

AttributionType = Literal[
    "DIRECT_QUOTE",
    "INDIRECT_SPEECH",
    "PARAPHRASED_POSITION",
    "PRONOUN_REFERENCE",
    "OTHER",
]


class AffiliatedOrg(BaseModel):
    affiliated_org_name: Optional[str] = Field(
        None,
        description="Canonical affiliated organization name, or null.",
    )
    affiliated_org_type: Optional[OrgType] = Field(
        None,
        description="Org ontology code for the affiliated organization, or null.",
    )
    affiliated_org_person_type: Optional[PersonType] = Field(
        None,
        description="Person type ontology code indicating the actor's role within that organization, or null.",
    )


class Actor(BaseModel):
    name: str = Field(
        ...,
        description="De-contextualised canonical name of the actor.",
    )
    entity_kind: EntityKind = Field(
        ...,
        description="Coarse entity kind of the actor.",
    )
    org_type: Optional[OrgType] = Field(
        None,
        description="Organisation type if actor is an organisation, else null.",
    )
    person_type: Optional[PersonType] = Field(
        None,
        description="Person ontology code if actor is a person, else null.",
    )
    role_or_title: Optional[str] = Field(
        None,
        description="Short free-text role/title, or null.",
    )
    affiliated_orgs: List[AffiliatedOrg] = Field(
        default_factory=list,
        description="List of affiliated organizations for the actor.",
    )


class Attribution(BaseModel):
    mention_in_article: Optional[str] = Field(
        None,
        description="How the actor is referred to in or near the span, or null if actor is assigned only via metadata.",
    )
    attribution_type: AttributionType = Field(
        ...,
        description="Type of attribution (direct quote, indirect speech, etc.).",
    )
    attribution_confidence: Literal[1, 2, 3, 4] = Field(
        ...,
        description="Confidence that this actor is the correct source for the span: 1=low, 4=high.",
    )


class DecontextualizedMention(BaseModel):
    mention_text: str = Field(
        ...,
        description="Original ambiguous or context-dependent mention.",
    )
    resolved_text: str = Field(
        ...,
        description="De-contextualised resolution of the mention.",
    )


class Content(BaseModel):
    mentions_to_decontextualize: List[DecontextualizedMention] = Field(
        default_factory=list,
        description="List of ambiguous mentions and their de-contextualised resolutions.",
    )
    central_claim: str = Field(
        ...,
        description="De-contextualised version of the statement.",
    )


class Statement(BaseModel):
    span_text: str = Field(
        ...,
        description="Exact text of the statement span as it appears in the text.",
    )
    span_char_start: Optional[int] = Field(
        None,
        description="Integer index of first character, or null if unknown.",
    )
    span_char_end: Optional[int] = Field(
        None,
        description="Integer index of last character (exclusive), or null if unknown.",
    )
    actor: Actor = Field(
        ...,
        description="Canonical actor entity responsible for the statement.",
    )
    attribution: Attribution = Field(
        ...,
        description="Attribution metadata linking the span to the actor.",
    )
    content: Content = Field(
        ...,
        description="De-contextualised content of the statement.",
    )
    reason: str = Field(
        ...,
        description="Brief explanation of why this span is one statement, how it was attributed, and any important de-contextualisation decisions.",
    )


class ArticleStatementsOutput(BaseModel):
    statements: List[Statement] = Field(
        default_factory=list,
        description="List of extracted statement objects.",
    )

In [ ]:
llm = ChatOpenAI(
    model="cyankiwi/Qwen3-Next-80B-A3B-Instruct-AWQ-4bit",
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
    temperature=0.0,
)
llm = llm.with_structured_output(ArticleStatementsOutput)

In [ ]:
PROMPT = ChatPromptTemplate([
    ("system", SYSTEM_PROMPT),
    ("human", """
Extract all statements from this article.
### Article
{article_text}
    """
    ),
])

In [ ]:
chain = PROMPT | llm

In [ ]:
articles = []
for jsonl_path in sorted(Path(JSONL_INPUT_DIR).glob("*.jsonl")):
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            article_content = f"""
    Content:
    {r['body']}

    Metadata:
    - source: {r['source_domain']}
    - source_type: NEWS_ARTICLE
    - date: {r['date']}
    - url: {r['url']}
    - language: {r['language']}
    - ai_tags: {', '.join(r.get('ai_tags', []))}
    - health_tags: {', '.join(r.get('health_tags', []))}
    """
            articles.append((r['id'], article_content))

print(f"{len(articles)} Artikel aus JSONL geladen")

In [ ]:
FAST_CONCURRENCY = 256     # main, high-throughput lane (keep GPU busy)
SLOW_CONCURRENCY = 64      # isolates pathological/slow requests

# Timeouts / thresholds (seconds)
FAST_TIMEOUT = 600         # if a request exceeds this in fast lane, we cancel & resubmit to slow lane
SLOW_TIMEOUT = 1800        # slow lane gets more time
TAIL_LOG_THRESHOLD = 60   # log any request slower than this

# Optional: cap generation length on the *client* to avoid stragglers
MAX_TOKENS = 4096
# how many total prompts you want to push (useful during iteration)
LIMIT = None  # e.g., 500 for a dry run, or None for all

In [ ]:
def rand_suffix(n=6):
    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=n))

def save_result(item_id: str, statement_id: str, data: dict):
    filename = f"{item_id}_{statement_id}_{rand_suffix()}.json"
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
import uuid

In [ ]:
import asyncio
import time
import uuid
import traceback
from dataclasses import dataclass, field
from typing import Any, Dict, List


@dataclass
class Progress:
    total: int
    done: int = 0
    ok: int = 0
    errors: int = 0
    slow_moved: int = 0
    fast_started: int = 0
    slow_started: int = 0
    started_at: float = field(default_factory=time.time)

    def line(self, queue_fast_size: int, queue_slow_size: int) -> str:
        elapsed = time.time() - self.started_at
        rate = self.done / elapsed if elapsed > 0 else 0.0
        remaining = self.total - self.done
        eta = remaining / rate if rate > 0 else float("inf")

        eta_str = f"{eta:.1f}s" if eta != float("inf") else "?"
        pct = 100 * self.done / self.total if self.total else 100.0

        return (
            f"[progress] {self.done}/{self.total} ({pct:.1f}%) | "
            f"ok={self.ok} err={self.errors} slow={self.slow_moved} | "
            f"q_fast={queue_fast_size} q_slow={queue_slow_size} | "
            f"rate={rate:.2f}/s eta={eta_str}"
        )


async def run_one_ainvoke(chain, item: Dict[str, Any], timeout_s: int) -> Any:
    t0 = time.time()
    try:
        res = await asyncio.wait_for(chain.ainvoke(item), timeout=timeout_s)
        dt = time.time() - t0
        if dt > TAIL_LOG_THRESHOLD:
            print(f"[tail] {dt:.1f}s")
        return res
    except asyncio.TimeoutError:
        raise
    except Exception:
        raise


async def run_all_with_slow_lane(
    chain,
    inputs: List[Dict[str, Any]],
    fast_conc: int = FAST_CONCURRENCY,
    slow_conc: int = SLOW_CONCURRENCY,
    fast_timeout: int = FAST_TIMEOUT,
    slow_timeout: int = SLOW_TIMEOUT,
):
    if LIMIT is not None:
        inputs = inputs[:LIMIT]

    results: List[Dict[str, Any]] = []
    errors: List[Dict[str, Any]] = []

    progress = Progress(total=len(inputs))
    progress_lock = asyncio.Lock()

    fast_sem = asyncio.Semaphore(fast_conc)
    slow_sem = asyncio.Semaphore(slow_conc)

    queue_fast = asyncio.Queue()
    queue_slow = asyncio.Queue()

    for item in inputs:
        queue_fast.put_nowait(item)

    async def mark_success(item_id, res_dict):
        async with progress_lock:
            results.append({"item_id": item_id, "result": res_dict})
            progress.done += 1
            progress.ok += 1

    async def mark_error(item, err_msg, trace=None):
        async with progress_lock:
            errors.append({"item": item, "error": err_msg, "trace": trace})
            print(item, err_msg)
            progress.done += 1
            progress.errors += 1

    async def fast_worker():
        while True:
            try:
                item = await queue_fast.get()
            except asyncio.CancelledError:
                return

            try:
                async with progress_lock:
                    progress.fast_started += 1

                async with fast_sem:
                    res = await run_one_ainvoke(
                        chain,
                        {"article_text": item[1]},
                        timeout_s=fast_timeout,
                    )

                res_dict = res.model_dump()
                save_result(item[0], uuid.uuid4(), res_dict)
                await mark_success(item[0], res_dict)

            except asyncio.TimeoutError:
                print("moved to slow lane")
                async with progress_lock:
                    progress.slow_moved += 1
                await queue_slow.put(item)

            except Exception as e:
                print("ERROR:", e)
                await mark_error(item, repr(e), traceback.format_exc())

            finally:
                queue_fast.task_done()

    async def slow_worker():
        while True:
            try:
                item = await queue_slow.get()
            except asyncio.CancelledError:
                return

            try:
                async with progress_lock:
                    progress.slow_started += 1

                async with slow_sem:
                    res = await run_one_ainvoke(
                        chain,
                        {"article_text": item[1]},
                        timeout_s=slow_timeout,
                    )

                res_dict = res.model_dump()
                save_result(item[0], uuid.uuid4(), res_dict)
                await mark_success(item[0], res_dict)

            except asyncio.TimeoutError:
                print("timed out")
                await mark_error(item, "Timeout in slow lane")

            except Exception as e:
                print("failed")
                await mark_error(item, repr(e), traceback.format_exc())

            finally:
                queue_slow.task_done()

    async def progress_reporter():
        last_line = None
        while True:
            async with progress_lock:
                line = progress.line(queue_fast.qsize(), queue_slow.qsize())
                finished = progress.done >= progress.total

            if line != last_line:
                print(line)
                last_line = line

            if finished:
                break

            await asyncio.sleep(1.0)

    fast_workers = [asyncio.create_task(fast_worker()) for _ in range(fast_conc)]
    slow_workers = [asyncio.create_task(slow_worker()) for _ in range(slow_conc)]
    reporter = asyncio.create_task(progress_reporter())

    await queue_fast.join()
    await queue_slow.join()
    await reporter

    for w in fast_workers + slow_workers:
        w.cancel()
    await asyncio.gather(*fast_workers, *slow_workers, return_exceptions=True)

    return results, errors

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
results, errors = asyncio.run(
    run_all_with_slow_lane(chain, articles,
                           fast_conc=FAST_CONCURRENCY,
                           slow_conc=SLOW_CONCURRENCY,
                           fast_timeout=FAST_TIMEOUT,
                           slow_timeout=SLOW_TIMEOUT)
)
